# Wrangling Harmonic-EEG Data

### Import libraries

First, import libraries necessary for exploring the data.

In [13]:
import pandas as pd
from pathlib import Path
import json

### Load in participant information

We decided to start with locating at the participants .tsv file from the downloaded data. From it, we saw that there are 40 participants (each labeled with their participant ID). Group stands for either 

In [10]:
# Load the TSV file into a DataFrame
participants_data = pd.read_csv('Data/participants.tsv', sep='\t')

# Convert to dataframe
df = pd.DataFrame(participants_data)

# Display the first few rows
df.head()

,participant_id,group,handedness,age,gender
0,sub-001,Cl,R,26,M
1,sub-002,Cl,R,26,M
2,sub-003,Cl,R,42,F
3,sub-004,Cl,R,24,F
4,sub-005,Imp,L,27,M


In [11]:

group_names = {"Cl": "classical musicians who don't improvise",
               "Imp": "improvisers (jazz-trained)"}

print(f"Participant ID (primary key) → {df['participant_id'].iloc[0]} ... "
      f"({df['participant_id'].nunique()} participants)")

print("Group:")
for g, n in df["group"].value_counts().items():
    print(f"  {g}: {group_names.get(g, g)} → {n}")

print(f"Age: mean: {df['age'].mean():.3f}, range: {df['age'].min()}–{df['age'].max()}")

sex_col = "sex" if "sex" in df.columns else "gender"
counts = df[sex_col].value_counts()
print("Gender: " + ", ".join(f"{k}: {v}" for k, v in counts.items()))

Participant ID (primary key) → sub-001 ... (40 participants)
Group:
  Imp: improvisers (jazz-trained) → 24
  Cl: classical musicians who don't improvise → 16
Age: mean: 25.275, range: 18–42
Gender: M: 24, F: 16


### Per-Participant file checking

In [15]:
root = Path("Data")

# ---------- 1. Inventory: which files does each participant have? ----------
rows = []
for d in sorted(root.glob("sub-*")):
    for file in (d / "eeg").glob("*"):
        rows.append({
            "participant": d.name,
            # "sub-001_task-AuditoryOddballChords_events.tsv" -> "events.tsv"
            "file": file.name.split("_")[-1],
            "size_MB": round(file.stat().st_size / 1e6, 2),
        })
rows = pd.DataFrame(rows)

# participants x file types table of sizes (NaN = missing)
presence = rows.pivot_table(index="participant", columns="file",
                            values="size_MB", aggfunc="first")

print(len(presence), "participants found")

print("\nFiles per type (should all equal the number of participants):")
print(presence.notna().sum())

missing = presence[presence.isna().any(axis=1)]
print("\nParticipants with missing files:")
print(missing if len(missing) else "  none")

# Tiny .set or .fdt files usually mean the file wasn't actually downloaded
print("\nSuspiciously small data files (<1 MB):")
small = presence.loc[(presence["eeg.fdt"] < 1) | (presence["eeg.set"] < 1),
                     ["eeg.set", "eeg.fdt"]]
print(small if len(small) else "  none")

40 participants found

Files per type (should all equal the number of participants):
file
channels.tsv        40
coordsystem.json    40
eeg.fdt             40
eeg.json            40
eeg.set             40
electrodes.tsv      40
events.json         40
events.tsv          40
dtype: int64

Participants with missing files:
  none

Suspiciously small data files (<1 MB):
  none


In [ ]:
""" 
Go through each participant and inspect the channels.tsv for each participant to
check for missing data, to produce one table for the channels.tsv for each participant
"""

_IncompleteInputError: incomplete input (2682125433.py, line 1)